# BPL Study Craft

Shared BPL setup for this folder and its subfolders. On startup, Craft checks PyPI and upgrades to the latest compatible stable release; no version is pinned. If an update replaces a version already loaded in an open kernel, restart that kernel before continuing. Run `setup_bpl()` to refresh the keyboard and toolbar, or `check_bpl_versions()` to list releases. Browser assets are regenerated from the installed package.


In [ ]:
# BEGIN BPL EXPRESSION EXPLORER SETUP
# BEGIN BPL EXPRESSION EXPLORER BACKEND
# This entire cell is self-contained. Remove it and the matching CRAFT.js block
# to uninstall Expression Explorer, then restart the dialogue kernel.
async def setup_bpl_explorer(*, start=True, dname=None):
    """Install the isolated explorer; start=False exposes its namespace for tests.

    Native BPL provides syntax and function construction. The capture hook
    observes normal cell runs; opening the explorer does not rerun a cell.
    All implementation details below stay local to this setup function.
    """
    global _bpl_expression_explorer_module
    previous = globals().get('_bpl_expression_explorer_module')
    if previous is not None:
        await previous.stop_explorer()

    import asyncio
    import bisect
    import functools
    import hashlib
    import importlib.metadata
    import inspect
    import math
    import re
    import time
    import types
    from collections import OrderedDict

    import basedpl as bp

    API_VERSION = 2
    CHANNEL_PREFIX = "bpl-explorer-v2-"
    MAX_SOURCE = 12000
    MAX_FUNCTION = 100
    MAX_CAPTURE_ITEMS = 4096
    MAX_SNAPSHOTS = 100
    TONES = ("blue", "purple", "green", "orange")
    # No names, system functions, literals, dfns, input, assignments or execution.
    PURE_GLYPHS = frozenset("+-×÷*⍟=≠≡≢<>≤≥⌈⌊|!○√~∧∨⍱⍲,⍪⍴↑↓⊂⊃∊∪∩⍳⍋⍒⌽⊖⍉⊢⊣/⌿\\⍀¨⍨∘⍥⊗⌸@() ")

    GLYPH_HELP = {
        "+": ("Plus", "One argument: conjugate. Two arguments: add. With /, it builds sum."),
        "/": ("Reduce", "Use the function on the left between items of the last axis. +/ sums a vector, or each row of a matrix."),
        "⌿": ("Reduce First", "Reduce along the first axis. On a matrix, +⌿ sums each column."),
        "~": ("Not / Without", "One argument: flip true and false. Two arguments: remove items of the right argument from the left."),
        "=": ("Equal", "Compare corresponding values. Each equal pair gives true; each different pair gives false."),
        "≠": ("Unique Mask / Not Equal", "One argument: mark first occurrences. Two arguments: compare corresponding values for inequality."),
        "≢": ("Tally / Not Match", "One argument: count leading-axis items. Two arguments: test whether the complete values differ, including shape."),
        "∘": ("Atop", "Build a composed function: apply the right function, then give its result to the left function."),
        "⌸": ("Key", "Build a grouping function. Group values by their corresponding keys, then call the operand once per group. One input uses its positions as values."),
        "⊗": ("Outer Product", "Apply the function on the left to every pair drawn from the two arguments."),
        ",": ("Ravel / Catenate", "One argument: flatten to a vector. Two arguments: join along the last axis."),
        "⍪": ("Table / Catenate First", "One argument: make a two-dimensional table. Two arguments: join along the first axis."),
        "←": ("Assignment", "Give the value or function on the right the name on the left."),
        "⍵": ("Right Argument", "The right argument supplied to this function call."),
        "⍺": ("Left Argument", "The left argument supplied to this function call."),
        "#": ("Replicate", "Repeat the items on the right using the counts on the left. Boolean counts filter items."),
        "⍴": ("Shape / Reshape", "One argument: report dimension lengths. Two arguments: reshape data using the lengths on the left."),
        "⍨": ("Self / Commute", "One argument: use it on both sides of the operand function. Two arguments: swap their positions."),
        "⍤": ("Rank", "Apply the operand function to cells of the rank specified on the right."),
        "@": ("Under", "For an invertible function on the right, transform the input, apply the left function, then undo the transformation."),
    }


    def source_body(source):
        """Only remove the notebook's first magic marker, preserving line offsets."""
        lines = str(source).replace("\r\n", "\n").split("\n")
        first = next((i for i, line in enumerate(lines) if line.strip()), None)
        if first is not None and lines[first].strip().split()[0] == "%%bpl":
            lines[first] = " " * len(lines[first])
        return "\n".join(lines)


    def source_key(source):
        return hashlib.sha256(source_body(source).strip().encode()).hexdigest()


    def _cell_id_from_parent(parent):
        """Decode SolveIt's observed cell-id.execution-id request-header format."""
        if not isinstance(parent, dict) or not isinstance(parent.get("header"), dict):
            return ""
        header = parent["header"]
        if header.get("msg_type") != "execute_request":
            return ""
        identifier = header.get("msg_id")
        if not isinstance(identifier, str):
            return ""
        match = re.fullmatch(r"([0-9a-fA-F]{8})\.[0-9a-fA-F]{8}", identifier)
        return match.group(1).lower() if match else ""


    def current_cell_id():
        try:
            from IPython import get_ipython
            return _cell_id_from_parent(get_ipython().kernel.get_parent())
        except Exception:
            return ""


    def _native(value):
        return getattr(value, "_inner", value)


    def function_tree(value):
        """Read BPL's assembled function, without calling it."""
        inner = _native(value)
        parts = inner.parts()
        if parts is None:
            return {"kind": "primitive", "text": str(inner), "children": []}
        kind, children = parts
        return {"kind": str(kind), "text": str(inner),
                "children": [function_tree(v) for v in children]}


    def _fingerprint(tree):
        if tree["kind"] == "primitive":
            return ("primitive", tree["text"])
        return (tree["kind"], tuple(_fingerprint(x) for x in tree["children"]))


    def _safe_function_text(text):
        return bool(text.strip()) and len(text) <= MAX_FUNCTION and "\n" not in text and "\r" not in text and all(c in PURE_GLYPHS or c.isspace() for c in text)


    def _json_value(value, budget=MAX_CAPTURE_ITEMS):
        """Copy bounded arrays only; preserve no live workspace object in a reply."""
        if hasattr(value, "py"):
            shape = getattr(value, "shape", ())
            if math.prod(shape) > budget:
                raise ValueError("The input is too large for an inline walkthrough.")
            value = value.py
        if hasattr(value, "tolist"):
            value = value.tolist()
        if value is None or isinstance(value, (str, bool, int)):
            if isinstance(value, str) and len(value) > budget:
                raise ValueError("The input is too large for an inline walkthrough.")
            return value
        if isinstance(value, float):
            return value if math.isfinite(value) else str(value)
        if isinstance(value, (list, tuple)):
            if len(value) > budget:
                raise ValueError("The input is too large for an inline walkthrough.")
            return [_json_value(v, budget // max(1, len(value))) for v in value]
        raise ValueError("This value does not have a supported walkthrough display.")


    def _node(start, end, text, label, description, tone="blue", children=None):
        return {"id": f"n-{start}-{end}", "span": [start, end], "text": text,
                "label": label, "title": label, "description": description,
                "tone": tone, "children": children or []}


    class Explorer:
        def __init__(self, workspace=None, cell_id_provider=None):
            self.workspace = workspace or bp.bpl
            self.private = bp._Workspace()
            self.private.timeout = 1
            self.version = str(getattr(bp, "__version__", None) or importlib.metadata.version("basedpl"))
            self.snapshots = OrderedDict()
            self._function_cache = {}
            self._capture_original = None
            self._capture_wrapper = None
            self._capturing = False
            self.cell_id_provider = cell_id_provider or (lambda: current_cell_id())
            self.epoch = str(time.time_ns())
            self.symbols = {x["glyph"]: x for x in getattr(bp, "symbols", [])}

        def compile_function(self, text):
            """Construct a function, never apply it to an argument."""
            text = text.strip()
            if not _safe_function_text(text):
                raise ValueError("This construction is outside the safe primitive subset.")
            if text not in self._function_cache:
                value = self.private(text)
                if not isinstance(value, bp.Function):
                    raise ValueError("BPL did not construct a function.")
                self._function_cache[text] = (value, function_tree(value))
                if len(self._function_cache) > 1024:
                    self._function_cache.pop(next(iter(self._function_cache)))
            return self._function_cache[text]

        def _token(self, native, line_source, line_start, n):
            a, b = native.span
            a, b = a - line_start, b - line_start
            text = line_source[a:b]
            kind = native.kind
            title, description = GLYPH_HELP.get(text, (None, None))
            if title is None:
                sym = self.symbols.get(text)
                if sym:
                    title = str(sym.get("name", kind)).replace("-", " ").title()
                    meanings = [str(sym.get(k, "")).replace("-", " ") for k in ("monad", "dyad")]
                    description = "; ".join(f"{lab}: {v}" for lab, v in zip(("One argument", "Two arguments"), meanings) if v) + "."
                else:
                    title = {"Name": "Name", "Literal": "Value", "Comment": "Comment", "Error": "Syntax Error"}.get(kind, kind)
                    description = {"Name": "A name whose value comes from the BPL workspace.", "Literal": "A value written directly in the expression.", "Comment": "A comment; it does not run.", "Error": "BPL's parser reports an error here."}.get(kind, f"BPL identifies this as {kind} syntax.")
            return {"id": f"t-{a}-{b}-{n}", "start": a, "end": b, "text": text,
                    "kind": kind, "title": title, "summary": description,
                    "description": description, "tone": TONES[n % len(TONES)]}

        def _spans_for_function(self, text, offset, tree):
            """Match native child functions against native-compiled source spans.

            This does not parse precedence. BPL supplies the tree. Every displayed
            semantic span must construct the same native function as that subtree.
            Ambiguous or missing span matches leave grouping unavailable.
            """
            syntax = bp.parse(text)
            boundaries = {0, len(text)}
            for node in syntax.walk():
                if node.kind not in ("Program", "Statement", "Comment"):
                    boundaries.update(node.span)
            boundaries = sorted(boundaries)
            candidates = []
            for start in boundaries:
                for end in boundaries:
                    if end <= start or not text[start:end].strip():
                        continue
                    try:
                        _, subtree = self.compile_function(text[start:end])
                        candidates.append((start, end, _fingerprint(subtree)))
                    except Exception:
                        pass

            def locate(t, bounds):
                if not t["children"] or t["kind"] not in ("fork", "∘"):
                    return [(t, bounds)]
                children = t["children"]
                choices = [[(a, b) for a, b, fp in candidates
                            if bounds[0] <= a < b <= bounds[1] and fp == _fingerprint(c)]
                           for c in children]
                matches = []

                def walk(i, prior, picked):
                    if i == len(choices):
                        matches.append(picked)
                        return
                    for pair in choices[i]:
                        if pair[0] >= prior:
                            walk(i + 1, pair[1], picked + [pair])
                walk(0, bounds[0], [])
                if not matches:
                    raise ValueError("The runtime tree has no unambiguous source span mapping.")
                # Parentheses and whitespace belong to the widest matching operand.
                matches.sort(key=lambda spans: sum(b - a for a, b in spans), reverse=True)
                best = matches[0]
                if len(matches) > 1 and sum(b-a for a,b in matches[1]) == sum(b-a for a,b in best) and matches[1] != best:
                    raise ValueError("Source span mapping is ambiguous.")
                return list(zip(children, best))

            return [(t, (a + offset, b + offset)) for t, (a, b) in locate(tree, (0, len(text)))]

        def _group_function(self, source, text, offset=0, assignment=None, valence=None):
            _, tree = self.compile_function(text)
            spans = self._spans_for_function(text, offset, tree)
            kind = tree["kind"]
            if kind == "fork":
                title = "Fork · 3 Functions"
                summary = "The left and right branches receive the original argument(s). Their two results become the left and right arguments of the middle function."
            elif kind == "∘":
                title = "Composition"
                summary = "The rightmost function runs first. Its result passes through each function to its left."
            else:
                title = "Derived Function" if tree["children"] else "Function"
                summary = "This is the function BPL constructs from these glyphs."
            nodes = []
            for i, (child, (a, b)) in enumerate(spans):
                glyphs = source[a:b]
                compact = "".join(glyphs.split()).strip("()")
                if kind == "fork":
                    label = ("Left Branch", "Middle Function", "Right Branch")[i]
                elif kind == "∘":
                    label = f"Step {len(spans) - i}"
                else:
                    label = {"/": "Reduce", "⌸": "Key", "⊗": "Outer Product"}.get(kind, "Function")
                # Valence follows the actual native tree, not a glyph's position in
                # raw text. Atop's left operand is monadic; a fork's middle operand
                # is dyadic. Other operands inherit their enclosing call's valence.
                child_valence = (1 if i == 0 else valence) if kind == "∘" else (2 if i == 1 else valence) if kind == "fork" else valence
                semantic_title = None
                semantic_summary = None
                if compact == "+/":
                    detail = "Sum: + and / combine into one function. With one argument, it sums the last axis."
                elif compact == "~":
                    if child_valence == 1:
                        semantic_title = "Not"
                        semantic_summary = "Flip true and false."
                        detail = "Here ~ receives one argument: it flips each Boolean, changing true to false and false to true."
                    elif child_valence == 2:
                        semantic_title = "Without"
                        semantic_summary = "Remove items found in the right argument from the left argument."
                        detail = "Here ~ receives two arguments: it removes items of the right argument from the left argument."
                    else:
                        semantic_title, detail = GLYPH_HELP["~"]
                        semantic_summary = detail
                else:
                    detail = GLYPH_HELP.get(compact, (None, "This operand is taken directly from BPL's assembled function tree."))[1]
                if kind == "fork":
                    detail += (" This branch receives the original argument(s)." if i != 1 else " This function receives the two branch results.")
                nested = []
                if child["kind"] in ("fork", "∘") and len(spans) > 1:
                    child_group = self._group_function(source, glyphs, a, valence=child_valence)
                    nested = child_group["nodes"]
                    detail = child_group["summary"]
                node = _node(a, b, glyphs, label, detail, TONES[i % 4], nested)
                if semantic_title:
                    node["annotation"] = {"title": semantic_title, "summary": semantic_summary, "description": detail}
                    node["valence"] = child_valence
                nodes.append(node)
            result = {"status": "exact", "kind": kind, "title": title, "summary": summary,
                      "description": summary, "nodes": nodes, "runtime_tree": tree,
                      "method": "BPL native function construction"}
            if assignment:
                result["assignment"] = assignment
                result["summary"] += f" The resulting function is assigned to {assignment['name']}."
            return result

        def _record_name(self, name):
            # A parser-supplied single Name is read by the original evaluator. It
            # cannot include a call, assignment, system function or user code.
            result = self._capture_original(name, {}, None, False)
            value = result.value
            if isinstance(value, bp.Function):
                tree = function_tree(value)
                return {"type": "function", "tree": tree, "text": str(value._inner)}
            return {"type": "array", "value": _json_value(value), "shape": list(getattr(value, "shape", ())) }

        def _take_snapshot(self, source, bindings):
            names = sorted({n.text for n in bp.parse(source).walk() if n.kind == "Name"})
            values = {}
            for name in names[:64]:
                try:
                    if name in bindings:
                        value = bindings[name]
                        values[name] = {"type": "array", "value": _json_value(value)}
                    else:
                        values[name] = self._record_name(name)
                except Exception:
                    pass
            return values

        def install_capture(self):
            """Observe normal execution; never trigger a notebook execution."""
            if self._capture_wrapper is not None:
                return
            prior = getattr(self.workspace._eval, "_bpl_explorer_original", self.workspace._eval)
            self._capture_original = prior
            owner = self

            @functools.wraps(prior)
            def capture(source, bindings=None, show=None, echo=False):
                bindings = bindings or {}
                if owner._capturing or not isinstance(source, str) or len(source) > MAX_SOURCE:
                    return prior(source, bindings, show, echo)
                owner._capturing = True
                before = {}
                try:
                    try:
                        before = owner._take_snapshot(source, bindings)
                    except Exception:
                        pass
                    record = {"source": source, "before": before, "time": time.time(), "error": None, "cell_id": owner.cell_id_provider()}
                    try:
                        result = prior(source, bindings, show, echo)
                        try:
                            record["result"] = _json_value(result.value)
                        except Exception:
                            pass
                        return result
                    except BaseException as exc:
                        record["error"] = str(exc)[:1200]
                        raise
                    finally:
                        try:
                            record["after"] = owner._take_snapshot(source, bindings)
                            key = (record["cell_id"], source_key(source))
                            owner.snapshots[key] = record
                            owner.snapshots.move_to_end(key)
                            while len(owner.snapshots) > MAX_SNAPSHOTS:
                                owner.snapshots.popitem(last=False)
                        except Exception:
                            pass
                finally:
                    owner._capturing = False

            capture._bpl_explorer_original = prior
            self._capture_wrapper = capture
            self.workspace._eval = capture

        def uninstall_capture(self):
            if self._capture_wrapper is not None and self.workspace._eval is self._capture_wrapper:
                self.workspace._eval = self._capture_original
            self._capture_wrapper = None

        def _walk_unavailable(self, message=None):
            return {"status": "unavailable", "message": message or "Run this cell once to record its inputs. This panel never reruns the cell.", "steps": []}

        def _pure(self, expression, **bindings):
            # Every caller supplies a fixed literal expression defined in this file.
            return self.private(expression, **bindings)

        def _dna_steps(self, text, left, right):
            """Known composed functions, validated against BPL's real function tree."""
            _, tree = self.compile_function(text)
            target = _fingerprint(tree)
            variants = {"match": "+/∘=", "difference": "+/~∘=", "unequal": "+/∘≠"}
            mode = next((k for k, v in variants.items() if _fingerprint(self.compile_function(v)[1]) == target), None)
            # Equivalent association (+/~)= has a different but valid tree.
            if mode is None and target == _fingerprint(self.compile_function("(+/~)=")[1]):
                mode = "difference"
            if mode is None or not isinstance(left, str) or not isinstance(right, str) or len(left) != len(right):
                return self._walk_unavailable("A value walkthrough is not available for this function and these recorded input types.")
            if len(left) > 160:
                return self._walk_unavailable("This input is too long for the compact value walkthrough.")
            compare = "≠" if mode == "unequal" else "="
            mask = _json_value(self._pure(f"lhs {compare} rhs", lhs=left, rhs=right))
            if not isinstance(mask, list):
                mask = [mask]
            rows = [{"label": "Left", "values": list(left)}, {"label": "Right", "values": list(right)},
                    {"label": "Different" if mode == "unequal" else "Equal", "values": mask, "highlight": mask}]
            steps = [{"id": "compare", "label": "Compare", "title": "Compare Corresponding Letters",
                      "description": f"{compare} receives both DNA vectors and produces one Boolean for each position.",
                      "tone": "green", "rows": rows}]
            if mode == "difference":
                mask = _json_value(self._pure("~mask", mask=mask))
                steps.append({"id": "not", "label": "Flip", "title": "Flip the Boolean Values",
                              "description": "~ receives the equality mask. Matches become false; differences become true.",
                              "tone": "purple", "rows": [{"label": "Different", "values": mask, "highlight": mask}]})
            total = _json_value(self._pure("+/mask", mask=mask))
            steps.append({"id": "sum", "label": "Sum", "title": "Add the True Values",
                          "description": "+/ adds the Boolean values, counting true as 1 and false as 0.",
                          "tone": "blue", "rows": [{"label": "Mask", "values": mask, "highlight": mask}],
                          "result": total, "result_label": "Matching Positions" if mode == "match" else "Different Positions"})
            return {"status": "exact", "message": "Values calculated by BPL from inputs recorded when this cell ran.", "steps": steps}

        def _outer_steps(self, left, right, operation, reduction):
            if not isinstance(left, str) or not isinstance(right, str) or len(left) > 12 or len(right) > 160:
                return self._walk_unavailable("The compact outer-product walkthrough supports recorded character vectors up to 12 by 160.")
            grid = _json_value(self._pure("lhs =⊗ rhs" if operation == "=" else "lhs ,⊗ rhs", lhs=left, rhs=right))
            steps = [{"id": "inputs", "label": "Inputs", "title": "Take One Item from Each Side",
                      "description": "The left vector supplies the row labels. Each position in the right vector supplies a column.",
                      "tone": "green", "rows": [{"label": "Left", "values": list(left)}, {"label": "Right", "values": list(right)}]}]
            rows = [{"label": str(label), "values": row, "highlight": row if operation == "=" else []} for label, row in zip(left, grid)]
            steps.append({"id": "pairs", "label": "Compare" if operation == "=" else "Join", "title": "Compare Every Pair" if operation == "=" else "Join Every Pair",
                          "description": "=⊗ compares every pair. A true value marks a match." if operation == "=" else ",⊗ joins every pair into a two-letter vector.",
                          "tone": "purple", "rows": rows})
            if reduction:
                counts = _json_value(self._pure("+/grid", grid=grid))
                steps.append({"id": "sum", "label": "Sum Rows", "title": "Count the Matches in Each Row",
                              "description": "+/ reduces the last axis, so it adds across each row. The totals stay in the left vector's order.",
                              "tone": "blue", "rows": [{"label": "Letter", "values": list(left)}, {"label": "Count", "values": counts}],
                              "result": counts, "result_label": "Counts"})
            return {"status": "exact", "message": "BPL calculations using the inputs recorded when this cell ran.", "steps": steps}

        def _key_steps(self, value, tally=False):
            if not isinstance(value, str) or len(value) > 100:
                return self._walk_unavailable("The compact Key walkthrough currently supports recorded character vectors up to 100 letters.")
            keys = _json_value(self._pure("(≠data)#data", data=value))
            indices = _json_value(self._pure("⍳≢data", data=value))
            positions = []
            for key in keys:
                positions.append(_json_value(self._pure("(data=letter)#indices", data=value, letter=key, indices=indices)))
            # The rows describe exact groups, obtained with BPL equality and replicate.
            steps = [{"id": "positions", "label": "Positions", "title": "Use Positions as the Values",
                      "description": "With one input, Key uses its items as keys and their zero-based positions as the values to group.",
                      "tone": "green", "rows": [{"label": "Letter", "values": list(value)}, {"label": "Position", "values": indices}]},
                     {"id": "groups", "label": "Group", "title": "Collect the Positions for Each Letter",
                      "description": "Each distinct letter keeps the positions where it occurs, in their original order.",
                      "tone": "purple", "rows": [{"label": key, "values": group} for key, group in zip(keys, positions)]}]
            if tally:
                counts = _json_value(self._pure("{≢ ⍵}⌸data", data=value))
                steps.append({"id": "operand", "label": "Count", "title": "Call Tally on Each Position Group",
                              "description": "Key calls the function once per letter. ⍵ receives that letter's positions; ≢ counts them.",
                              "tone": "blue", "rows": [{"label": "Letter", "values": list(keys)}, {"label": "Count", "values": counts}],
                              "result": counts, "result_label": "Counts"})
            else:
                rows = _json_value(self._pure(",⌸data", data=value))
                steps.append({"id": "operand", "label": "Join", "title": "Join Each Letter to Its Positions",
                              "description": "For each group, comma receives the letter on the left and its positions on the right. Key collects the returned rows.",
                              "tone": "blue", "rows": [{"label": key, "values": [key] + group} for key, group in zip(keys, positions)]})
            return {"status": "exact", "message": "BPL calculations using the input recorded when this cell ran.", "steps": steps}

        def _known_application(self, line, items, snapshot):
            """Narrow, documented forms; array arguments must be captured values."""
            source = line["source"]
            before = snapshot.get("before", {}) if snapshot else {}
            texts = [n.text for n in items]
            reduction = texts[:2] == ["+", "/"]
            body = items[2:] if reduction else items
            body_texts = [n.text for n in body]
            if len(body) == 4 and body[0].kind == body[3].kind == "Name" and body_texts[1] in ("=", ",") and body_texts[2] == "⊗":
                left, op, _, right = body_texts
                if left not in before or right not in before or before[left].get("type") != "array" or before[right].get("type") != "array":
                    return False
                function = op + "⊗"
                _, tree = self.compile_function(function)
                a, b = body[1].span[0], body[2].span[1]
                nodes = [_node(*body[0].span, left, "Left Argument", "The recorded vector supplies one row per item.", "green"),
                         _node(a, b, source[a:b], "Outer Product", "BPL applies this function to every left/right pair.", "purple"),
                         _node(*body[3].span, right, "Right Argument", "The recorded vector supplies one column per item.", "green")]
                if reduction:
                    nodes.insert(0, _node(items[0].span[0], items[1].span[1], source[items[0].span[0]:items[1].span[1]], "Sum Rows", "Sum the last axis of the outer-product result.", "blue"))
                line["grouping"] = {"status": "exact", "kind": "outer-product", "title": "Outer Product" + (" Then Sum" if reduction else ""),
                    "summary": "Form every pair from the two vectors, then apply the operand function to each pair." + (" +/ sums the resulting rows." if reduction else ""),
                    "nodes": nodes, "runtime_tree": tree, "method": "Native function and recorded array arguments; supported application form"}
                line["walkthrough"] = self._outer_steps(before[left]["value"], before[right]["value"], op, reduction)
                return True
            compact = "".join(n.text for n in items).replace(" ", "")
            # No arbitrary dfn is executed: only these two literal operand bodies.
            if items and items[-1].kind == "Name":
                name = items[-1].text
                operand = "".join(n.text for n in items[:-1]).replace(" ", "")
                if operand in (",⌸", "{≢⍵}⌸") and before.get(name, {}).get("type") == "array":
                    tally = operand.startswith("{")
                    function = "{≢ ⍵}⌸" if tally else ",⌸"
                    constructed = self.private(function)  # fixed library string only
                    tree = function_tree(constructed)
                    if tree["kind"] != "⌸":
                        return False
                    start, end = items[0].span[0], items[-2].span[1]
                    line["grouping"] = {"status": "exact", "kind": "key", "title": "Key with One Input",
                        "summary": "Group the input's zero-based positions by item, then call the operand once per group.",
                        "nodes": [_node(start, end, source[start:end], "Grouping Function", "⌸ modifies the function on its left. Each call receives a key and its position group.", "blue"),
                                  _node(*items[-1].span, name, "Keys", "The input supplies the keys. Its positions supply the grouped values.", "green")],
                        "runtime_tree": tree, "method": "Native function and recorded array argument; supported Key form"}
                    line["walkthrough"] = self._key_steps(before[name]["value"], tally)
                    return True
            return False

        def _analyse_line(self, line, parsed, snapshot, local_functions):
            source = line["source"]
            nodes = list(parsed.children)
            statements = [n for n in nodes if n.kind == "Statement"]
            if len(statements) != 1:
                return
            def run_items(nodes):
                for node in nodes:
                    if node.kind == "Run":
                        yield from run_items(node.children)
                    else:
                        yield node
            # Run is the parser's adjacent-glyph container. Keep actual explicit
            # Group, Dfn and array nodes intact when exposing its constituent tokens.
            items = list(run_items(statements[0].children))
            assignment = None
            if len(items) >= 3 and items[0].kind == "Name" and items[1].kind == "Assign":
                assignment = {"name": items[0].text, "span": list(items[0].span), "arrow_span": list(items[1].span)}
                line["_assigned_name"] = items[0].text
                local_functions.pop(items[0].text, None)
                items = items[2:]
            if not items:
                return
            offset, end = items[0].span[0], items[-1].span[1]
            expression = source[offset:end]
            if _safe_function_text(expression):
                try:
                    line["grouping"] = self._group_function(source, expression, offset, assignment)
                    if assignment:
                        local_functions[assignment["name"]] = expression
                    return
                except Exception as exc:
                    line["grouping"]["message"] = str(exc)
                    return

            if self._known_application(line, items, snapshot):
                return

            # A simple dyadic named-function call can use a function captured during
            # this exact execution. Its internals are not guessed from the name.
            if len(items) == 3 and all(n.kind == "Name" for n in items):
                lhs, func, rhs = [n.text for n in items]
                before = snapshot.get("before", {}) if snapshot else {}
                if before.get(lhs, {}).get("type") != "array" or before.get(rhs, {}).get("type") != "array":
                    return
                ftext = local_functions.get(func)
                saved_func = before.get(func, {})
                if ftext is None and saved_func.get("type") == "function":
                    possible = saved_func.get("text", "")
                    if _safe_function_text(possible):
                        try:
                            _, check = self.compile_function(possible)
                            if _fingerprint(check) == _fingerprint(saved_func["tree"]):
                                ftext = possible
                        except Exception:
                            pass
                if ftext is not None:
                    descriptions = ["The recorded left argument.", "The function assembled by BPL.", "The recorded right argument."]
                    line["grouping"] = {"status": "exact", "kind": "dyadic-call", "title": "Dyadic Function Call",
                        "summary": f"{func} receives {lhs} on the left and {rhs} on the right.",
                        "nodes": [_node(*n.span, n.text, label, description, tone) for n, label, description, tone in zip(items, ("Left Argument", "Function", "Right Argument"), descriptions, TONES)],
                        "runtime_tree": self.compile_function(ftext)[1], "method": "Captured native function"}
                    if lhs in before and rhs in before and before[lhs]["type"] == before[rhs]["type"] == "array":
                        line["walkthrough"] = self._dna_steps(ftext, before[lhs]["value"], before[rhs]["value"])
                        line["_function_name"] = func
                    return

            # Keep the grouping honest for general applications: explicit parser
            # groups are available, while implicit binding is not exported by parse.
            explicit = [n for n in items if n.kind in ("Group", "Dfn", "ArrayLiteral")]
            if explicit:
                line["grouping"]["nodes"] = [_node(*n.span, n.text, "Explicit " + n.kind, "This boundary is reported by BPL's parser.", TONES[i % 4]) for i, n in enumerate(explicit)]

        def inspect_source(self, source, cell_id=None):
            if not isinstance(source, str) or len(source) > MAX_SOURCE:
                return {"version": self.version, "status": "error", "message": "Choose a BPL cell shorter than 12,000 characters.", "lines": []}
            body = source_body(source)
            snapshot = self.snapshots.get((str(cell_id or "").removeprefix("_").removesuffix("-i"), source_key(source)))
            split = body.split("\n")
            starts, pos = [], 0
            for line in split:
                starts.append(pos)
                pos += len(line) + 1
            tree = bp.parse(body)
            # A cell-level snapshot cannot tell us an intermediate value after a
            # reassignment. Never borrow the value from before the cell in that case.
            if snapshot:
                snapshot = dict(snapshot, before=dict(snapshot.get("before", {})))
                for statement in tree.walk():
                    children = list(statement.children)
                    for j, child in enumerate(children[:-1]):
                        if child.kind == "Name" and children[j + 1].kind == "Assign":
                            snapshot["before"].pop(child.text, None)
            multiline = [n.span for n in tree.walk() if n.kind not in ("Program", "Comment") and "\n" in n.text]
            leaves = [n for n in tree.walk() if not n.children and n.kind not in ("Program", "Statement", "Body", "Cell")]
            lines = []
            local_functions = {}
            for i, text in enumerate(split):
                if not text.strip():
                    continue
                line = {"source": text, "line_number": i + 1, "tokens": [],
                        "grouping": {"status": "unavailable", "kind": "syntax", "title": "Syntax Only",
                            "message": "BPL's parser provides these tokens and explicit brackets. The runtime does not expose a complete source-mapped call tree for this expression.", "nodes": []},
                        "walkthrough": self._walk_unavailable()}
                for node in leaves:
                    a, b = node.span
                    if starts[i] <= a < b <= starts[i] + len(text):
                        # Fail closed if a library changes its span convention.
                        if body[a:b] != node.text:
                            continue
                        line["tokens"].append(self._token(node, text, starts[i], len(line["tokens"])))
                # Delimiters are exact source characters, not inferred precedence.
                covered = {j for token in line["tokens"] for j in range(token["start"], token["end"])}
                for j, char in enumerate(text):
                    if char in "()[]{}" and j not in covered:
                        line["tokens"].append({"id": f"t-{j}-{j+1}", "start": j, "end": j+1, "text": char,
                            "kind": "Delimiter", "title": "Explicit Bracket", "description": "A bracket written in the source.", "tone": "orange"})
                line["tokens"].sort(key=lambda t: (t["start"], t["end"]))
                errors = [n for n in leaves if n.kind == "Error" and n.span[0] < starts[i] + len(text) and n.span[1] > starts[i]]
                if errors:
                    error = errors[0]
                    line["grouping"]["message"] = str(getattr(error, "error", "BPL syntax error"))
                    line["walkthrough"] = self._walk_unavailable("BPL reports a syntax error in this expression.")
                elif not any(a < starts[i] + len(text) and b > starts[i] for a, b in multiline):
                    try:
                        self._analyse_line(line, bp.parse(text), snapshot, local_functions)
                    except Exception as exc:
                        line["grouping"]["message"] = f"Grouping unavailable: {str(exc)[:240]}"
                if snapshot and snapshot.get("error"):
                    line["execution_error"] = snapshot["error"]
                    line["walkthrough"] = self._walk_unavailable("The recorded cell run ended with an error. Its error is shown without inventing later results.")
                lines.append(line)
            # Let a function-definition line use the captured call immediately below
            # it; this is the same execution, not fabricated example inputs.
            for index, line in enumerate(lines):
                assignment = line["grouping"].get("assignment")
                if assignment:
                    for following in lines[index + 1:]:
                        if following.get("_assigned_name") == assignment["name"]:
                            break
                        if following.get("_function_name") == assignment["name"]:
                            line["walkthrough"] = following["walkthrough"]
                            break
            for line in lines:
                line.pop("_function_name", None)
                line.pop("_assigned_name", None)
            return {"version": self.version, "api_version": API_VERSION, "status": "ok", "source_hash": source_key(source),
                    "capture": {"status": "recorded" if snapshot else "not-recorded", "time": snapshot.get("time") if snapshot else None},
                    "message": "Native BPL syntax and function construction. Unsupported runtime grouping is labelled explicitly.", "lines": lines}


    _active = None


    async def start_explorer(ipython=None, dname=None):
        """Start a dialogue-local relay and observe subsequent BPL cell runs."""
        nonlocal _active
        import dialoghelper.core as dh
        await stop_explorer()
        name = str(dname or dh.find_dname()).lstrip("/").removesuffix(".ipynb")
        channel_name = CHANNEL_PREFIX + hashlib.sha256(name.encode()).hexdigest()[:24]
        explorer = Explorer()
        explorer.install_capture()
        try:
            channel = await dh.Channel.connect(channel_name)
        except Exception:
            explorer.uninstall_capture()
            raise

        async def listen():
            while True:
                event = await channel.events.get()
                op = event.get("op")
                if op == "hello":
                    await channel.send(op="ready", version=explorer.version, api_version=API_VERSION, epoch=explorer.epoch)
                elif op == "inspect":
                    request_id = str(event.get("request_id", ""))[:200]
                    try:
                        result = explorer.inspect_source(event.get("source", ""), event.get("cell_id"))
                        result["request_source_hash"] = event.get("source_hash")
                    except Exception as exc:
                        result = {"version": explorer.version, "status": "error", "message": str(exc)[:600], "lines": []}
                        result["request_source_hash"] = event.get("source_hash")
                    await channel.send(op="inspection", request_id=request_id, epoch=explorer.epoch, version=explorer.version, result=result)

        task = asyncio.create_task(listen(), name="bpl-expression-explorer")
        _active = {"explorer": explorer, "channel": channel, "task": task, "name": name}
        return {"version": explorer.version, "channel": channel_name, "capture": "installed"}


    async def stop_explorer():
        nonlocal _active
        active, _active = _active, None
        if active is None:
            return
        active["explorer"].uninstall_capture()
        active["task"].cancel()
        try:
            await active["task"]
        except asyncio.CancelledError:
            pass
        await active["channel"].close()

    class ExplorerNamespace(types.SimpleNamespace):
        @property
        def _active(self):
            return _active

    _bpl_expression_explorer_module = ExplorerNamespace(
        Explorer=Explorer, start_explorer=start_explorer,
        stop_explorer=stop_explorer, current_cell_id=current_cell_id,
        _cell_id_from_parent=_cell_id_from_parent,
        source_body=source_body, source_key=source_key,
        function_tree=function_tree, API_VERSION=API_VERSION,
    )
    if not start:
        return _bpl_expression_explorer_module
    result = await start_explorer(dname=dname)
    print('Expression explorer ready (BPL ' + result['version'] + ').')
    return None

await setup_bpl_explorer()
# END BPL EXPRESSION EXPLORER BACKEND
# END BPL EXPRESSION EXPLORER SETUP


In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import importlib
import json
import subprocess
import sys
from IPython import get_ipython
from IPython.display import HTML, Javascript, display
from dialoghelper.core import realpath

def ensure_latest_bpl():
    """Check PyPI and install the latest compatible stable BPL release."""
    loaded = sys.modules.get('basedpl')
    running = getattr(loaded, '__version__', None)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--upgrade',
        '--index-url', 'https://pypi.org/simple',
        '--disable-pip-version-check', '--no-input', '--quiet', 'basedpl'
    ])
    importlib.invalidate_caches()
    installed = version('basedpl')
    if loaded is not None and running != installed:
        raise RuntimeError(
            f'BPL {installed} is installed, but this kernel has BPL {running or "an older version"} loaded. '
            'Restart this dialogue kernel, then run the dialogue again to load the updated Craft and BPL.'
        )
    print(f'BPL {installed}: latest compatible stable release checked on PyPI.')
    return installed

def check_bpl_versions():
    """List installed and available BPL releases without changing packages."""
    return subprocess.check_call([
        sys.executable, '-m', 'pip', 'index', 'versions', 'basedpl',
        '--index-url', 'https://pypi.org/simple', '--disable-pip-version-check'
    ])

_bpl_version = ensure_latest_bpl()
if get_ipython().find_line_magic('bpl') is None:
    get_ipython().run_line_magic('load_ext', 'basedpl.notebooks')
_bpl_folder = Path(await realpath(''))
_bpl_craft_dir = next(p for p in [_bpl_folder, *_bpl_folder.parents] if (p / 'CRAFT.js').is_file())


BPL 0.1.67: latest compatible stable release checked on PyPI.


In [ ]:
bpl_examples = {'⍳': [{'label': 'One input · zero-based indices', 'code': '⍳4'}, {'label': 'Two inputs · index of', 'code': '"abc"⍳"cabz"'}], '↦': [{'label': 'One input · range from one', 'code': '↦4'}, {'label': 'Two inputs · inclusive range', 'code': '0↦4'}], '=': [{'label': 'One input · classify equal items', 'code': '=4 7 4 9'}, {'label': 'Two inputs · compare each item', 'code': '1 2 3 = 1 9 3'}], '≠': [{'label': 'One input · first occurrences', 'code': '≠4 7 4 9'}, {'label': 'Two inputs · not equal', 'code': '1 2 3 ≠ 1 9 3'}], '≡': [{'label': 'One input · nesting depth', 'code': '≡1 2 3'}, {'label': 'Two inputs · whole-array match', 'code': '1 2 3 ≡ 1 2 3'}], '≢': [{'label': 'One input · count items', 'code': '≢2 4 9'}, {'label': 'Two inputs · whole-array mismatch', 'code': '1 2 3 ≢ 1 2'}], '⍴': [{'label': 'One input · shape', 'code': '⍴1 2 3'}, {'label': 'Two inputs · reshape', 'code': '2 3⍴⍳6'}], '×': [{'label': 'One input · sign', 'code': '×¯3 0 4'}, {'label': 'Two inputs · multiply', 'code': '3×1 2 3'}], '÷': [{'label': 'One input · reciprocal', 'code': '÷1 2 4'}, {'label': 'Two inputs · divide', 'code': '12÷3'}], '+': [{'label': 'Two inputs · add', 'code': '1 2 3 + 10'}], '-': [{'label': 'One input · negate', 'code': '-1 2 3'}, {'label': 'Two inputs · subtract', 'code': '10-3'}], '/': [{'label': 'Reduce · sum a list', 'code': '+/2 4 9'}, {'label': 'Reduce · product', 'code': '×/1 2 3 4'}], '\\': [{'label': 'Scan · running sum', 'code': '+\\1 2 3 4'}], '⌽': [{'label': 'One input · reverse', 'code': '⌽1 2 3'}, {'label': 'Two inputs · rotate', 'code': '1⌽1 2 3'}], '↑': [{'label': 'One input · first', 'code': '↑10 20 30'}, {'label': 'Two inputs · take', 'code': '2↑10 20 30'}], '↓': [{'label': 'Two inputs · drop', 'code': '1↓10 20 30'}], '⎕': [{'label': 'Display a result', 'code': '⎕←1 2 3'}], '⍟': [{'label': 'One input · natural logarithm', 'code': '⍟1 2 4'}, {'label': 'Two inputs · logarithm base two', 'code': '2⍟8'}], '√': [{'label': 'One input · square root', 'code': '√1 4 9'}, {'label': 'Two inputs · cube root', 'code': '3√27'}]}

In [ ]:
def _bpl_keyboard():

    # Run after loading basedpl.notebooks to enable keyboard symbol selection.
    import json
    from importlib.resources import files
    from basedpl import symbols
    from IPython import get_ipython
    from IPython.display import HTML, Javascript, display

    def _patch(source, old, new):
        if source.count(old) != 1: raise RuntimeError(f'Keyboard integration needs updating for installed BPL; expected one match for {old!r}.')
        return source.replace(old, new)

    _assets = files('basedpl')
    _lb = (_assets / 'lb.js').read_text()
    _input = (_assets / 'input.js').read_text()
    _lb = _patch(_lb, 'let overlay = false, active, lastEditor, choice, keyInput = false;', '''let overlay = false, active, lastEditor, choice, keyInput = false, picked = 0;
        function highlight() {
            tip.setAttribute('role', 'listbox');
            [...tip.querySelectorAll('button')].forEach((b, i) => {
                b.setAttribute('role', 'option'); b.setAttribute('aria-selected', String(i === picked));
            });
        }''')
    _lb = _patch(_lb, 'tip.replaceChildren();', 'picked = 0; tip.replaceChildren(); tip.scrollTop = 0;')
    _lb = _patch(_lb, 'tip.hidden = false;', 'tip.hidden = false; if (!anchor) highlight();')
    _lb = _patch(_lb, 'const typed = active?.id === e.id && active.start === item?.start;', '''const typed = active?.id === e.id && active.start === item?.start;
            if (typed && plain && !ev.shiftKey && !tip.hidden && !choice?.anchor &&
                ['ArrowUp', 'ArrowDown', 'Enter', 'Tab', 'Escape'].includes(ev.key)) {
                const buttons = [...tip.querySelectorAll('button')];
                if (ev.key === 'Escape') cancel();
                else if (ev.key === 'ArrowUp' || ev.key === 'ArrowDown') {
                    if (buttons.length) picked = (picked + (ev.key === 'ArrowDown' ? 1 : -1) + buttons.length) % buttons.length;
                    highlight(); buttons[picked]?.scrollIntoView({block:'nearest'});
                } else {
                    if (buttons.length) e.insert(buttons[picked].dataset.glyph, item.start);
                    cancel();
                }
                ev.preventDefault(); ev.stopImmediatePropagation(); return;
            }''')
    _input = _patch(_input, r'^%%(?:bpl|apl)[^\S\n]*', r'^%%(?:bpl|apl)(?:[ \t]+--animate)?[^\S\n]*')
    display(Javascript(f"{_lb}({json.dumps(symbols)}, {_input}, {(_assets / 'layout.json').read_text()})"))
    display(HTML('<style>' + (_assets / 'fonts.css').read_text() +
                 '.bpl_choices button[aria-selected="true"]{background:#2463a8;color:white}</style>'))
    get_ipython().find_line_magic('bpl').__self__._loaded = True

    display(Javascript(r'''(()=>{
    const bar=document.querySelector('.ngn_lb'); if(bar) bar.hidden=false;
    let button=document.getElementById('bpl-symbol-toggle');
    if(!button){
        button=document.createElement('button'); button.id='bpl-symbol-toggle';
        button.type='button'; button.textContent='BPL';
        button.title='Show or hide BPL symbols';
        button.className='uk-btn uk-btn-sm uk-btn-default cursor-pointer';
        document.getElementById('settings').before(button);
    }
    button.setAttribute('aria-pressed',String(bar && !bar.hidden));
    button.onclick=()=>{
        const current=document.querySelector('.ngn_lb');
        if(current){current.hidden=!current.hidden;button.setAttribute('aria-pressed',String(!current.hidden));}
    };
    })();'''))



In [ ]:
def rebuild_bpl_browser():
    """Regenerate the native browser bootstrap after changing keyboard code or examples."""
    from unittest.mock import patch
    from basedpl import symbols
    from importlib.resources import files
    captured = []
    with patch('IPython.display.display', lambda obj: captured.append(obj)):
        _bpl_keyboard()
    scripts = [obj.data for obj in captured if isinstance(obj, Javascript)]
    styles = ''.join(obj.data.removeprefix('<style>').removesuffix('</style>') for obj in captured if isinstance(obj, HTML))
    layout = json.loads((files('basedpl') / 'layout.json').read_text())
    payload = ','.join(json.dumps(x, ensure_ascii=False) for x in (symbols, layout, bpl_examples))
    marker = '// BEGIN GENERATED BPL BOOTSTRAP'
    target = _bpl_craft_dir / 'CRAFT.js'
    base = target.read_text().split(marker)[0].rstrip()
    boot = '\n' + marker + '\n(() => {\n'
    boot += 'const start=()=>{\n'
    boot += 'if(!document.getElementById("bpl-craft-fonts")){const s=document.createElement("style");s.id="bpl-craft-fonts";s.textContent=' + json.dumps(styles) + ';document.head.append(s);}\n'
    boot += '\n;\n'.join(scripts) + '\n;\n'
    boot += '((symbols,layout,examples)=>window.bplCraftPalette(window.bplCraftCatalogue(symbols,layout),examples))(' + payload + ');\n'
    boot += '};if(document.readyState==="loading")document.addEventListener("DOMContentLoaded",start,{once:true});else start();\n})();\n'
    target.write_text(base + '\n' + boot)


In [ ]:
def setup_bpl():
    """Refresh the BPL keyboard and palette without changing lesson variables."""
    from basedpl import symbols
    from importlib.resources import files
    _bpl_keyboard()
    script = (_bpl_craft_dir / 'CRAFT.js').read_text().split('// BEGIN GENERATED BPL BOOTSTRAP')[0]
    layout = json.loads((files('basedpl') / 'layout.json').read_text())
    size = dict(fontSize=19, buttonWidth=21, buttonHeight=26, gap=0.5)
    payload = ','.join(json.dumps(x, ensure_ascii=False) for x in (symbols, layout, bpl_examples, size))
    script += '\n;((symbols,layout,examples,size)=>{window.bplCraftPalette(window.bplCraftCatalogue(symbols,layout),examples,size);})(' + payload + ');'
    display(Javascript(script))
    get_ipython().find_line_magic('bpl').__self__._loaded = True

rebuild_bpl_browser()
setup_bpl()
